# Examples

> Two example treemaps: a protein interactome grouped by gene family, and the world's countries by region.

In [ ]:
#| default_exp treemap.examples


In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *
import collections, csv, re
from pathlib import Path

The examples are built here from two small tables in the repository's `data/treemap/` folder,
and packaged with the library as `itemiset/data/treemap_examples.json`.

**DDX6 partners by gene group.** The 42 high-confidence BioID preys of the DDX6 bait in Youn
J-Y *et al.* (2018) *Molecular Cell* 69:517–532
([doi:10.1016/j.molcel.2017.12.020](https://doi.org/10.1016/j.molcel.2017.12.020)), from the
BioGRID file the Euler examples use (release 5.0.262, MIT License), grouped by
[HGNC gene group](https://www.genenames.org/data/genegroup/) (HGNC data are released under
CC0). Proteins are matched to HGNC by Entrez Gene ID, so names BioGRID still lists under
retired symbols appear under their current ones (RQCD1 as CNOT9, for example).

**Countries by UN region.** Every country and territory with at least a million people,
grouped by UN M49 continent and subregion, with its population in millions. From the Unicode
Common Locale Data Repository (CLDR) release 48.2.3, under the Unicode License v3, whose notice
is kept beside the data in `data/treemap/CLDR-LICENSE.txt`.

## How the tables were made

This cell is not run by the tests: it needs the full HGNC and CLDR downloads, listed in
`data/README.md` with their checksums. It is kept so the tables can be rebuilt.

In [ ]:
#| eval: false
import collections, csv, gzip, json
from pathlib import Path

def extract_hgnc_groups(biogrid_gz, hgnc_dir, out_csv, bait="DDX6"):
    "One row per (prey, candidate gene group): the bait's preys, their HGNC records, and every gene group they belong to, ancestors included."
    hgnc_dir = Path(hgnc_dir)
    csv.field_size_limit(10 ** 8)
    A, B = "Official Symbol Interactor A", "Official Symbol Interactor B"
    preys = {}
    for r in csv.DictReader(gzip.open(biogrid_gz, "rt"), delimiter="\t"):
        if r[A] == bait:
            preys[r[B]] = r["Entrez Gene Interactor B"]
    hgnc = {r["entrez_id"]: r for r in csv.DictReader(open(hgnc_dir / "hgnc_complete_set.txt"), delimiter="\t")
            if r["entrez_id"]}
    fam_name = {r["id"]: r["name"] for r in csv.DictReader(open(hgnc_dir / "family.csv"))}
    parents = collections.defaultdict(set)
    for r in csv.DictReader(open(hgnc_dir / "hierarchy.csv")):
        parents[r["child_fam_id"]].add(r["parent_fam_id"])
    gene_fams = collections.defaultdict(set)
    for r in csv.DictReader(open(hgnc_dir / "gene_has_family.csv")):
        gene_fams[r["hgnc_id"]].add(r["family_id"])

    def with_ancestors(fams):
        out, stack = set(fams), list(fams)
        while stack:
            for p in parents.get(stack.pop(), ()):
                if p not in out:
                    out.add(p)
                    stack.append(p)
        return out

    size = collections.Counter(f for fs in gene_fams.values() for f in with_ancestors(fs))
    rows = []
    for prey in sorted(preys):
        rec = hgnc[preys[prey]]
        hid = rec["hgnc_id"].split(":")[-1]
        fams = sorted(with_ancestors(gene_fams.get(hid, ())), key=int)
        base = dict(biogrid_symbol=prey, entrez_id=preys[prey], hgnc_id=rec["hgnc_id"], symbol=rec["symbol"])
        if not fams:
            rows.append({**base, "group_id": "", "group_name": "", "group_size": ""})
        for f in fams:
            rows.append({**base, "group_id": f, "group_name": fam_name[f], "group_size": size[f]})
    with open(out_csv, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0]), lineterminator="\n")
        w.writeheader()
        w.writerows(rows)
    return len(preys), len(rows)

def extract_countries(cldr_dir, out_csv):
    "Every ISO 3166-1 territory in CLDR's UN M49 tree, with its continent, subregion, English short name and population."
    cldr_dir = Path(cldr_dir)
    info = json.load(open(cldr_dir / "territoryInfo.json"))["supplemental"]["territoryInfo"]
    contain = json.load(open(cldr_dir / "territoryContainment.json"))["supplemental"]["territoryContainment"]
    names = json.load(open(cldr_dir / "territories.json"))["main"]["en"]["localeDisplayNames"]["territories"]
    not_iso = {"AC", "CP", "CQ", "DG", "EA", "IC", "TA", "XK"}   # CLDR codes outside ISO 3166-1
    kids = lambda code: contain.get(code, {}).get("_contains", [])
    name = lambda code: names.get(f"{code}-alt-short", names.get(code, code))
    rows = []
    for continent in kids("001"):   # World -> Africa, Americas, Asia, Europe, Oceania
        stack = [(continent, None)]
        while stack:
            region, sub = stack.pop()
            for c in kids(region):
                if c.isdigit():
                    stack.append((c, c))
                elif len(c) == 2 and c in info and c not in not_iso:
                    rows.append(dict(continent=name(continent), subregion=name(sub or region), code=c,
                                     country=name(c), population=int(float(info[c]["_population"]))))
    rows.sort(key=lambda r: (r["continent"], r["subregion"], r["country"]))
    with open(out_csv, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0]), lineterminator="\n")
        w.writeheader()
        w.writerows(rows)
    return len(rows)

# HGNC: https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt and
# .../hgnc/csv/csv/genefamily_db_tables/{family,hierarchy,gene_has_family}.csv (downloaded 22 Oct 2026)
# CLDR: https://raw.githubusercontent.com/unicode-org/cldr-json/48.2.3/cldr-json/... (see data/README.md)
extract_hgnc_groups("../../data/youn2018_biogrid-5.0.262.tab3.txt.gz", "hgnc_downloads",
                    "../../data/treemap/ddx6_hgnc_groups.csv")
extract_countries("cldr_48.2.3", "../../data/treemap/cldr_m49_countries.csv")

## Loading an example

`load_example` returns the arguments for `plot`, so an example draws in one line:
`plot(**load_example("countries"))`. Change any of them on the way, e.g.
`plot(**{**load_example("ddx6"), "title": "My title"})`.

In [ ]:
#| export
import json
from importlib.resources import files

In [ ]:
#| export
def load_example(name: str = "ddx6") -> dict:
    "The `plot` arguments for a bundled example: `\"ddx6\"` (proteins by gene group) or `\"countries\"` (by UN region)."
    data = json.loads(files("itemiset").joinpath("data/treemap_examples.json").read_text(encoding="utf-8"))
    if name not in data:
        raise KeyError(f"No example called {name!r}; choose from {sorted(data)}")
    ex = dict(data[name])
    ex["last"] = tuple(ex.get("last", ()))
    return ex

## DDX6 partners by gene group

HGNC puts many genes in several gene groups at once (CNOT4 is in five), and groups nest
(Argonaute RISC component family sits inside Argonaute/PIWI family). A treemap needs one place
per protein, so each gets one group by a fixed rule:

1. A protein's candidates are its gene groups and all the groups that contain them.
2. Groups defined by structure or genomic position ("… domain containing", "… repeat …",
   "Zinc fingers …", "… host genes") count only if a protein has nothing else.
3. Repeatedly take the group covering the most proteins not yet placed (ties: the smaller group
   in HGNC, then the name) and place those proteins in it.
4. Groups left holding a single protein are pooled as "Other gene groups"; proteins in no gene
   group go in "No HGNC group".

In [ ]:
root = Path("../..")
rows = list(csv.DictReader(open(root/"data/treemap/ddx6_hgnc_groups.csv", encoding="utf-8")))
STRUCTURAL = re.compile(r"domain containing|host genes|^Zinc fingers|repeat", re.I)
OTHER, NONE = "Other gene groups", "No HGNC group"

cands, size, gname = collections.defaultdict(set), {}, {}
for r in rows:
    cands[r["symbol"]]
    if r["group_id"]:
        cands[r["symbol"]].add(r["group_id"])
        size[r["group_id"]], gname[r["group_id"]] = int(r["group_size"]), r["group_name"]
for p, fs in cands.items():
    functional = {f for f in fs if not STRUCTURAL.search(gname[f])}
    cands[p] = functional or fs

group_of, todo = {}, {p for p, fs in cands.items() if fs}
while todo:
    cover = collections.Counter(f for p in todo for f in cands[p])
    best = min(cover, key=lambda f: (-cover[f], size[f], gname[f]))
    for p in sorted(todo):
        if best in cands[p]:
            group_of[p] = gname[best]
    todo -= set(group_of)
members = collections.Counter(group_of.values())
ddx6 = [dict(group=(NONE if p not in group_of else group_of[p] if members[group_of[p]] > 1 else OTHER), protein=p)
        for p in sorted(cands)]
collections.Counter(r["group"] for r in ddx6).most_common()

[('Other gene groups', 13),
 ('CCR4-NOT transcription complex', 8),
 ('No HGNC group', 6),
 ('RNA decapping and exonuclease complex subunits', 5),
 ('Argonaute RISC component family', 3),
 ('LSm proteins', 3),
 ('RNA binding motif containing', 2),
 ('SURF complex', 2)]

In [ ]:
counts = collections.Counter(r["group"] for r in ddx6)
test_eq(len(ddx6), 42)
test_eq(counts, {"CCR4-NOT transcription complex": 8, "RNA decapping and exonuclease complex subunits": 5,
                 "Argonaute RISC component family": 3, "LSm proteins": 3, "RNA binding motif containing": 2,
                 "SURF complex": 2, OTHER: 13, NONE: 6})
test_eq({r["protein"] for r in ddx6} >= {"CNOT9", "MARF1", "MCRIP2", "TUT4"}, True)   # current symbols

## Countries by UN region

In [ ]:
countries = [dict(continent=r["continent"], subregion=r["subregion"], country=r["country"],
                  population_millions=int(int(r["population"]) / 1e6 + 0.5))
             for r in csv.DictReader(open(root/"data/treemap/cldr_m49_countries.csv", encoding="utf-8"))
             if int(r["population"]) >= 1_000_000]
test_eq(len(countries), 159)
test_eq(collections.Counter(r["continent"] for r in countries),
        {"Africa": 49, "Asia": 47, "Europe": 35, "Americas": 25, "Oceania": 3})
test_eq(max(countries, key=lambda r: r["population_millions"])["country"], "China")

## Packaging

The examples are written as `plot` arguments. Short header labels replace HGNC's long group
names; a soft hyphen (`­`) marks where a word may break if its header must go on two lines.

In [ ]:
examples = {
    "ddx6": dict(
        data=ddx6, levels=["group"], name="protein", numeral=None,
        labels={"CCR4-NOT transcription complex": "CCR4–NOT complex",
                "RNA decapping and exonuclease complex subunits": "De­capping complex",
                "Argonaute RISC component family": "Argo­nautes",
                "RNA binding motif containing": "RNA binding motif"},
        last=[OTHER, NONE],
        title="DDX6 proximity partners by gene group",
        subtitle="The 42 high-confidence BioID preys of DDX6, grouped by HGNC gene group.",
        note=("Data: Youn et al. (2018) Mol Cell 69:517, doi:10.1016/j.molcel.2017.12.020, via BioGRID 5.0.262 "
              "(MIT License). Gene groups: HGNC, genenames.org (CC0); one group per protein.")),
    "countries": dict(
        data=countries, levels=["continent", "subregion"], name="country", numeral="population_millions",
        labels={}, last=[],
        title="The world's countries by UN region",
        subtitle=("Countries and territories with at least 1 million people, by UN M49 continent and "
                  "subregion. Number = population, millions."),
        note="Data: Unicode CLDR 48.2.3 (Unicode License v3): population and UN M49 regions."),
}

def write_if_changed(path, text):
    path = Path(path)
    if path.exists() and path.read_text(encoding="utf-8") == text: return False
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    return True

write_if_changed(root/"itemiset/data/treemap_examples.json", json.dumps(examples, ensure_ascii=False, indent=1) + "\n")

False

In [ ]:
ex = load_example("ddx6")
test_eq(ex["levels"], ["group"])
test_eq(len(ex["data"]), 42)
test_eq(ex["last"], (OTHER, NONE))
test_fail(lambda: load_example("nope"), contains="countries")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()